# Nipah Virus — Named Entity Recognition (NER)

---

| | |
|---|---|
| **Mata Kuliah** | COMP6885001 — Natural Language Processing |
| **Topik** | Information Extraction menggunakan spaCy Rule-Based NER |
| **Entitas** | `DISEASE`, `SYMPTOM`, `LOCATION` |
| **Dataset** | Web scraping berita + dataset manual (300 kalimat) |
| **Model** | spaCy EntityRuler (rule-based, CPU-friendly) |

---

## Latar Belakang

Virus Nipah (NiV) adalah virus zoonosis berbahaya yang pertama kali ditemukan di Malaysia tahun 1998. Penyakit ini dapat menyebabkan ensefalitis (radang otak) hingga kematian dengan tingkat fatalitas mencapai 40–75%.

Proyek ini membangun sistem **Named Entity Recognition (NER)** untuk mengekstrak informasi penting dari teks berita dan artikel tentang virus Nipah secara otomatis:
- 🔴 **DISEASE** — nama penyakit/virus (contoh: *virus Nipah, ensefalitis, NiV*)
- 🟡 **SYMPTOM** — gejala klinis (contoh: *demam, kejang, sesak napas*)
- 🔵 **LOCATION** — lokasi kejadian (contoh: *Malaysia, Kerala, Bangladesh*)

---

## Alur Pipeline

```
1. Install & Import
2. Web Scraping Berita  ──┐
3. Load Dataset Lama   ──┴──▶  4. Gabung & Deduplikasi
                                        │
                                        ▼
                               5. Preprocessing
                                        │
                                        ▼
                           6. NER spaCy Rule-Based
                                        │
                                        ▼
                           7. Evaluasi (P / R / F1)
                                        │
                                        ▼
                          8. Simpan Model & Dataset
```

---
## 1. Install & Import

Library yang digunakan:

| Library | Fungsi |
|---|---|
| `spacy` | NLP pipeline & EntityRuler untuk rule-based NER |
| `en_core_web_sm` | Model bahasa Inggris kecil dari spaCy |
| `requests` + `beautifulsoup4` | Web scraping artikel berita |
| `pandas` | Manipulasi dan penyimpanan dataset |
| `re` | Preprocessing teks dengan regex |

In [ ]:
!pip install -q spacy requests beautifulsoup4 lxml
!python -m spacy download en_core_web_sm -q
!pip install -q datasets
!pip install -q wikipedia-api
print('Semua library berhasil diinstall!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 93.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 9.5 MB/s eta 0:00:00
Semua library berhasil diinstall!


In [ ]:
import os
import pandas as pd
import requests
import re
import time
import spacy
import wikipediaapi
import unicodedata

from collections import Counter
from datasets import load_dataset
from spacy.pipeline import EntityRuler
from bs4 import BeautifulSoup
from collections import defaultdict
from spacy import displacy


print('Semua library berhasil diimport!')

Semua library berhasil diimport!


---
## 2. Web Scraping Berita

Untuk memperbanyak data, dilakukan **web scraping** otomatis dari Google News RSS Feed.

### Mengapa Web Scraping?
Dataset yang dikumpulkan secara manual memiliki keterbatasan jumlah dan variasi kalimat. Web scraping memungkinkan kita mendapatkan data yang lebih banyak, lebih beragam, dan mencerminkan penggunaan bahasa yang nyata (*real-world text*).

### Sumber Data
- **Google News RSS** — snippet judul dan deskripsi berita dari berbagai media
- Query menggunakan kata kunci **Bahasa Indonesia dan Inggris** untuk cakupan yang lebih luas

> **Catatan etika:** Data hanya mengambil snippet publik dari RSS feed (bukan full article) sehingga tidak melanggar *terms of service*.

In [ ]:
def scrape_google_news(query, max_articles=20):
    """
    Scraping snippet berita dari Google News RSS Feed.
    - Tidak memerlukan API key
    - Gratis dan legal (mengakses feed publik)
    """
    query_encoded = query.replace(' ', '+')
    url = f'https://news.google.com/rss/search?q={query_encoded}&hl=id&gl=ID&ceid=ID:id'
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'xml')
        items = soup.find_all('item')
    except Exception as e:
        print(f'Gagal scraping "{query}": {e}')
        return []

    results = []
    for item in items[:max_articles]:
        title = item.find('title').text if item.find('title') else ''
        desc  = item.find('description').text if item.find('description') else ''
        raw   = title + '. ' + desc
        clean = BeautifulSoup(raw, 'html.parser').get_text()
        clean = re.sub(r'\s+', ' ', clean).strip()
        if len(clean) > 30:
            results.append(clean)
    return results

In [ ]:
# Daftar query — kombinasi Bahasa Indonesia & Inggris
queries = [
    'virus nipah',
    'nipah virus outbreak',
    'virus nipah gejala',
    'nipah virus symptoms',
    'wabah nipah malaysia india',
    'nipah encephalitis',
    'virus nipah penularan kelelawar',
]

scraped_texts = []
for q in queries:
    texts = scrape_google_news(q, max_articles=20)
    scraped_texts.extend(texts)
    print(f'  ✔ Query "{q}" → {len(texts)} artikel')
    time.sleep(1)  # jeda agar tidak diblokir server

print(f'\nTotal teks terkumpul dari scraping: {len(scraped_texts)}')

  ✔ Query "virus nipah" → 20 artikel
  ✔ Query "nipah virus outbreak" → 20 artikel
  ✔ Query "virus nipah gejala" → 20 artikel
  ✔ Query "nipah virus symptoms" → 20 artikel
  ✔ Query "wabah nipah malaysia india" → 20 artikel
  ✔ Query "nipah encephalitis" → 20 artikel
  ✔ Query "virus nipah penularan kelelawar" → 20 artikel

Total teks terkumpul dari scraping: 140


In [ ]:
def split_sentences(texts):
    """
    Memecah teks panjang menjadi kalimat-kalimat individual.
    Kalimat kurang dari 20 karakter dibuang (biasanya noise).
    """
    sentences = []
    for text in texts:
        parts = re.split(r'(?<=[.!?])\s+', text)
        for part in parts:
            part = part.strip()
            if len(part) > 20:
                sentences.append(part)
    return sentences

scraped_sentences = split_sentences(scraped_texts)
print(f'Total kalimat dari scraping: {len(scraped_sentences)}')
print('\nContoh kalimat hasil scraping:')
for s in scraped_sentences[:3]:
    print(f'  - {s}')

Total kalimat dari scraping: 302

Contoh kalimat hasil scraping:
  - Apa Itu Virus Nipah yang Berpotensi Jadi Ancaman Kesehatan Global?
  - Ini Gejala dan Bahayanya - Mitra Keluarga.
  - Apa Itu Virus Nipah yang Berpotensi Jadi Ancaman Kesehatan Global?


---
## 3. Load Dataset yang Lama (Manual)

Dataset ini berisi **300 kalimat** yang dikumpulkan secara manual dari berbagai artikel dan berita tentang virus Nipah. Dataset ini akan digabungkan dengan hasil scraping untuk membentuk dataset yang lebih lengkap.

>  `Dataset Nipah.csv`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive berhasil terhubung!')

Mounted at /content/drive
Google Drive berhasil terhubung!


In [ ]:
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if 'nipah' in file.lower() or 'Nipah' in file:
            print(os.path.join(root, file))

/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/nipah_ner_model.zip
/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/model_A_nipah_NER_v2(tdk_bagus).ipynb
/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset_Nipah_PubMed_Geo.ipynb
/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/model_A_nipah_NER.ipynb
/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Dataset Nipah.csv
/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Dataset_Nipah_Final.gsheet
/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Dataset_Nipah_Final.csv
/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Dataset_Nipah_PubMed_Geo.csv
/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Requirements Docs/Named Entity Recognition for Disease and Symptom Extraction in Nipah Virus Text us

In [ ]:
PATH_DATASET= '/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Dataset Nipah.csv'

In [ ]:
# Sesuaikan path ini dengan lokasi file di Google Drive kamu
data = []
with open(PATH_DATASET, 'r', encoding='utf-8', errors='replace') as f:
    f.readline()  # skip baris header
    for line in f:
        line = line.strip()
        if line:
            parts = line.split(',', 1)
            if len(parts) == 2:
                data.append(parts[1])

df_lama = pd.DataFrame({'text': data})
print(f'Dataset sebelumnya berhasil diload: {len(df_lama)} kalimat')
df_lama.head(5)

Dataset sebelumnya berhasil diload: 300 kalimat


,text
0,Virus nipah (NiV) adalah virus zoonosis yang b...
1,Pertama kali diidentifikasi pada tahun 1998 di...
2,Menurut data dari World Health Organization (W...
3,"People with infection can develop a fever, and..."
4,Fruit bats of the Pteropodidae family are the ...


---
## Load Dataset Kaggle — Disease Symptom Description

Dataset ini didownload dari **Kaggle** (`itachi9604/disease-symptom-description-dataset`) dan disimpan di Google Drive.

File yang digunakan: `symptom_Description.csv` — berisi nama penyakit dan deskripsi gejala dalam bentuk kalimat.

> Path: `.../Dataset/Kaggle/symptom_Description.csv`

In [ ]:
PATH_KAGGLE = '/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Kaggle/symptom_Description.csv'

df_kaggle = pd.read_csv(PATH_KAGGLE)
print(f'Kolom tersedia: {df_kaggle.columns.tolist()}')
print(f'Total baris   : {len(df_kaggle)}')
df_kaggle.head(5)

Kolom tersedia: ['Disease', 'Description']
Total baris   : 41


,Disease,Description
0,Drug Reaction,An adverse drug reaction (ADR) is an injury ca...
1,Malaria,An infectious disease caused by protozoan para...
2,Allergy,An allergy is an immune system response to a f...
3,Hypothyroidism,"Hypothyroidism, also called underactive thyroi..."
4,Psoriasis,Psoriasis is a common skin disorder that forms...


In [ ]:
# Ambil kolom deskripsi
desc_col = [c for c in df_kaggle.columns if 'escri' in c or 'desc' in c.lower()][0]
print(f'Kolom deskripsi: {desc_col}')

kaggle_sentences = split_sentences(df_kaggle[desc_col].dropna().astype(str).tolist())
df_kaggle_final = pd.DataFrame({'text': kaggle_sentences})

print(f'Total kalimat dari Kaggle: {len(df_kaggle_final)}')
df_kaggle_final.head(5)

Kolom deskripsi: Description
Total kalimat dari Kaggle: 105


,text
0,An adverse drug reaction (ADR) is an injury ca...
1,ADRs may occur following a single dose or prol...
2,An infectious disease caused by protozoan para...
3,Falciparum malaria is the most deadly type.
4,An allergy is an immune system response to a f...


---
## Web Scraping Wikipedia

Wikipedia menyediakan artikel lengkap tentang Virus Nipah dalam **Bahasa Indonesia dan Inggris**.

Kelebihan sumber ini:
- Konten terstruktur dan terverifikasi secara ensiklopedi
- Mencakup informasi medis, gejala, lokasi wabah, dan sejarah penyebaran
- Kredibel untuk disebutkan sebagai sumber di laporan akademis

> Scraping dilakukan menggunakan library `wikipedia-api`.

In [ ]:
def scrape_wikipedia(judul, bahasa='id'):
    """
    Scraping artikel Wikipedia dan memecahnya menjadi kalimat.
    bahasa: 'id' = Indonesia, 'en' = Inggris
    """
    wiki = wikipediaapi.Wikipedia(language=bahasa, user_agent='NipahNER/1.0')
    page = wiki.page(judul)
    if not page.exists():
        print(f'  Halaman "{judul}" tidak ditemukan')
        return []
    sentences = split_sentences([page.text])
    print(f'  ✔ Wikipedia [{bahasa}] "{judul}" → {len(sentences)} kalimat')
    return sentences

wiki_sentences = []
wiki_sentences += scrape_wikipedia('Virus Nipah', 'id')
wiki_sentences += scrape_wikipedia('Nipah virus infection', 'en')
wiki_sentences += scrape_wikipedia('Nipah virus', 'en')

df_wiki = pd.DataFrame({'text': wiki_sentences})
print(f'\nTotal kalimat dari Wikipedia: {len(df_wiki)}')
df_wiki.head(5)

  ✔ Wikipedia [id] "Virus Nipah" → 12 kalimat
  ✔ Wikipedia [en] "Nipah virus infection" → 119 kalimat
  ✔ Wikipedia [en] "Nipah virus" → 121 kalimat

Total kalimat dari Wikipedia: 252


,text
0,Virus Nipah yang teridentifikasi pada tahun 19...
1,"Di Singapura, 11 kasus termasuk satu kematian ..."
2,Virus Nipah diklasifikasikan oleh CDC sebagai ...
3,Kemunculan\nVirus Nipah telah diisolasi dari k...
4,lylei dan kelelawar daun bulat Horsfield (Hipp...


---
##  Web Scraping WHO & CDC

Halaman resmi **WHO** dan **CDC** menyediakan informasi medis yang akurat dan terverifikasi tentang Virus Nipah.

| Sumber | URL |
|---|---|
| WHO | https://www.who.int/news-room/fact-sheets/detail/nipah-virus |
| CDC | https://www.cdc.gov/nipah/about/index.html |

> Konten dari situs resmi organisasi kesehatan dunia memperkuat kualitas dan kredibilitas dataset.

In [ ]:
def scrape_website(url, tag='p'):
    """
    Scraping paragraf teks dari website.
    tag: HTML tag yang berisi teks utama (biasanya 'p')
    """
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(resp.content, 'html.parser')
        paragraphs = [p.get_text() for p in soup.find_all(tag) if len(p.get_text()) > 30]
        sentences = split_sentences(paragraphs)
        print(f'  {url[:55]}... → {len(sentences)} kalimat')
        return sentences
    except Exception as e:
        print(f'Gagal scraping {url}: {e}')
        return []

who_cdc_sentences = []
who_cdc_sentences += scrape_website('https://www.who.int/news-room/fact-sheets/detail/nipah-virus')
who_cdc_sentences += scrape_website('https://www.cdc.gov/nipah/about/index.html')

df_who_cdc = pd.DataFrame({'text': who_cdc_sentences})
print(f'\nTotal kalimat dari WHO + CDC: {len(df_who_cdc)}')
df_who_cdc.head(5)

  https://www.who.int/news-room/fact-sheets/detail/nipah-... → 57 kalimat
  https://www.cdc.gov/nipah/about/index.html... → 9 kalimat

Total kalimat dari WHO + CDC: 66


,text
0,"Nipah virus is a zoonotic virus, usually trans..."
1,Nipah virus was first identified in 1998 durin...
2,"In 1999, an outbreak was reported in Singapore..."
3,No new outbreaks have been reported from Malay...
4,"In 2001, Nipah virus infection outbreaks were ..."


# DataSet Jurnal

In [ ]:
# 1. Load dataset PubMed yang baru saja kamu save di Drive
PATH_PUBMED = '/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Dataset_Nipah_PubMed_Geo.csv'
df_pubmed_baru = pd.read_csv(PATH_PUBMED)

# 2. (Opsional) Jika kamu menjalankan ini di notebook baru,
# load juga dataset lama kamu:
PATH_LAMA = '/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Dataset_Nipah_Final.csv'
df_gabung = pd.read_csv(PATH_LAMA)

df_gabung.head(5)

,id,text_clean,lang,text_en,silver_DISEASE,silver_LOCATION
0,1,Virus nipah (NiV) adalah virus zoonosis yang b...,id,"Nipah virus (NiV) is a zoonotic virus, which m...","['##pa', 'ni', '##v']",[]
1,2,Pertama kali diidentifikasi pada tahun 1998 di...,id,It was first identified in 1998 in Malaysia du...,[],['Malaysia']
2,3,Menurut data dari World Health Organization (W...,id,According to data from the World Health Organi...,"['##pa', 'infections', 'en', '##cephalitis', '...",[]
3,4,"People with infection can develop a fever, and...",en,"People with infection can develop a fever, and...",['infection'],[]
4,5,Fruit bats of the Pteropodidae family are the ...,en,Fruit bats of the Pteropodidae family are the ...,['pt'],[]


---
## 4. Gabungkan & Deduplikasi Dataset

Seluruh sumber data digabungkan menjadi satu dataset komprehensif:

| No | Sumber | Keterangan |
|---|---|---|
| 1 | Dataset manual | 300 kalimat dari berita |
| 2 | Google News RSS | Scraping berita terkini |
| 3 | Kaggle | Deskripsi gejala penyakit |
| 4 | Wikipedia | Artikel Virus Nipah ID + EN |
| 5 | WHO + CDC | Halaman resmi organisasi kesehatan dunia |
| 6 | Artikel | Nipah |

Setelah digabung, dilakukan **deduplikasi** untuk menghapus kalimat yang sama persis.

In [ ]:
df_scraping = pd.DataFrame({'text': scraped_sentences})

# Gabungkan semua 5 sumber
df_gabung = pd.concat([
    df_lama,          # 1. Dataset manual
    df_scraping,      # 2. Google News scraping
    df_kaggle_final,  # 3. Kaggle disease symptom
    df_wiki,          # 4. Wikipedia
    df_who_cdc,       # 5. WHO + CDC
    df_pubmed_baru,   # 6. PubMed
], ignore_index=True)

print(f'\nJumlah sebelum deduplikasi : {len(df_gabung)}')
print(f'   1. Dataset manual           : {len(df_lama)}')
print(f'   2. Google News scraping      : {len(df_scraping)}')
print(f'   3. Kaggle disease symptom    : {len(df_kaggle_final)}')
print(f'   4. Wikipedia                 : {len(df_wiki)}')
print(f'   5. WHO + CDC                 : {len(df_who_cdc)}')
print(f'   6. Artikel/Jurnal            : {len(df_pubmed_baru)}')

# Deduplikasi & bersihkan
df_gabung['text'] = df_gabung['text'].str.strip()
df_gabung = df_gabung.drop_duplicates(subset='text')
df_gabung = df_gabung[df_gabung['text'].str.len() > 20]
df_gabung = df_gabung.reset_index(drop=True)
df_gabung['id'] = df_gabung.index + 1
df_gabung = df_gabung[['id', 'text']]

print(f'\nJumlah setelah deduplikasi  : {len(df_gabung)} kalimat')
df_gabung.head()


Jumlah sebelum deduplikasi : 5317
   1. Dataset manual           : 300
   2. Google News scraping      : 302
   3. Kaggle disease symptom    : 105
   4. Wikipedia                 : 252
   5. WHO + CDC                 : 66
   6. Artikel/Jurnal            : 4292

Jumlah setelah deduplikasi  : 5185 kalimat


,id,text
0,1,Virus nipah (NiV) adalah virus zoonosis yang b...
1,2,Pertama kali diidentifikasi pada tahun 1998 di...
2,3,Menurut data dari World Health Organization (W...
3,4,"People with infection can develop a fever, and..."
4,5,Fruit bats of the Pteropodidae family are the ...
